# Image crop stability

Timestamp: 2026-09-12 00:02:43 +04:00


## Hypothesis

Absolutely positioning each `.food-photo` at `inset: 0` with 100% width and height, `object-fit: cover`, and centered object positioning makes every loaded image fill and centrally crop to the `.photo-panel` box without allowing intrinsic image dimensions to affect panel layout.


## Method

Extract CSS from `../index.html` and assert the panel containment and complete image fitting declarations, including the absence of image `min-height` in all media contexts. Execute the unchanged inline JavaScript in Node with a mocked DOM over 60 deterministic clicks to verify 20 choices, no consecutive repeats, matching photo assignment, and code point 160 before the emoji. Syntax-check the extracted inline JavaScript with `node --check`.


In [1]:
import re
from pathlib import Path
html = Path("../index.html").read_text()
css = re.search(r"<style>([\s\S]*?)</style>", html).group(1)
panel = re.search(r"\.photo-panel\s*\{([^}]*)\}", css).group(1)
food_rules = re.findall(r"\.food-photo\s*\{([^}]*)\}", css)
food = "\n".join(food_rules)
for pattern in (r"position:\s*relative", r"overflow:\s*hidden"):
    assert re.search(pattern, panel)
assert len(food_rules) == 1
for pattern in (r"position:\s*absolute", r"inset:\s*0", r"width:\s*100%", r"height:\s*100%", r"object-fit:\s*cover", r"object-position:\s*center"):
    assert re.search(pattern, food)
assert "min-height" not in food
print("CSS_CHECK PASS")
print("photo_panel=position:relative,overflow:hidden")
print("food_photo_rules=1")
print("food_photo=position:absolute,inset:0,width:100%,height:100%,object-fit:cover,object-position:center")
print("food_photo_min_height=absent")


CSS_CHECK PASS
photo_panel=position:relative,overflow:hidden
food_photo_rules=1
food_photo=position:absolute,inset:0,width:100%,height:100%,object-fit:cover,object-position:center
food_photo_min_height=absent


In [2]:
import subprocess
node_check = r'''const fs=require("fs"),vm=require("vm"),assert=require("assert");
const html=fs.readFileSync("../index.html","utf8");
const scripts=[...html.matchAll(/<script>([\s\S]*?)<\/script>/g)];
assert.strictEqual(scripts.length,1);
let click;
const make=()=>({classList:{add(){},remove(){}},hidden:false,offsetWidth:0});
const elements={"#generate-button":{addEventListener(type,fn){if(type==="click")click=fn;}},"#result":make(),"#food-photo":make(),"#photo-placeholder":make(),"#photo-credit":make()};
const values=Array.from({length:60},(_,i)=>((i*7)%20)/20);
let n=0;
const mockMath=Object.create(Math); mockMath.random=()=>values[n++%values.length];
const context={document:{querySelector:s=>elements[s]},Math:mockMath};
vm.createContext(context);
vm.runInContext(scripts[0][1]+"\nglobalThis.__opts=lunchOptions;",context);
assert.strictEqual(context.__opts.length,20);
const picks=[];
for(let i=0;i<60;i++){
  click();
  const text=elements["#result"].textContent;
  const choice=context.__opts.find(o=>text===`Today\x27s pick: ${o.name}!\u00a0🥳`);
  assert(choice);
  assert.strictEqual(elements["#food-photo"].src,choice.photo);
  assert.strictEqual(elements["#food-photo"].alt,`A plate of ${choice.name}`);
  const emoji=text.indexOf("🥳");
  assert.strictEqual(text.charCodeAt(emoji-1),160);
  picks.push(choice.name);
}
assert(picks.every((x,i)=>i===0||x!==picks[i-1]));
console.log("JS_MOCK_CHECK PASS");
console.log(`choices=${context.__opts.length}`);
console.log(`clicks=${picks.length} consecutive_duplicates=0`);
console.log("matched_photo_assignments=60");
console.log("code_point_before_emoji=160");'''
completed = subprocess.run(["node", "-e", node_check], check=True, text=True, capture_output=True)
print(completed.stdout, end="")


JS_MOCK_CHECK PASS
choices=20
clicks=60 consecutive_duplicates=0
matched_photo_assignments=60
code_point_before_emoji=160


In [3]:
import re
import subprocess
from pathlib import Path
html = Path("../index.html").read_text()
scripts = re.findall(r"<script>([\s\S]*?)</script>", html)
assert len(scripts) == 1
subprocess.run(["node", "--check"], input=scripts[0], text=True, check=True, capture_output=True)
print("INLINE_JS_SYNTAX PASS")


INLINE_JS_SYNTAX PASS


## Interpretation

The captured CSS check confirms `.photo-panel` remains a positioned, overflow-hidden containing box and the sole `.food-photo` rule supplies absolute centered cover fitting with no image `min-height`. The mocked JavaScript check retained all 20 choices, matched every tested image URL to its selected dish, produced no consecutive repeat in 60 clicks, and retained NBSP code point 160 before the emoji. Inline JavaScript syntax passed.
